# Culture Scan

**Imports and Initialisation**

In [1]:
# !pip install --upgrade google-cloud-vertexai
from google.cloud import storage, bigquery, secretmanager
from google.api_core import exceptions
import google.generativeai as genai # Corrected import from google.genai to google.generativeai
import os
import io
import time
import random
import json
import re
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from tqdm import tqdm
from google.colab import userdata


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [2]:
start_time = time.time()
print(f"🚀 Starting process at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


🚀 Starting process at 2026-04-13 07:47:45


**Configuration**

In [3]:
def access_secret_version(project_id: str, secret_id: str, version_id: str = "latest") -> str:
    """Retrieve a secret payload from Google Secret Manager."""
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    try:
        response = client.access_secret_version(request={"name": name})
        return response.payload.data.decode("UTF-8")
    except Exception as e:
        print(f"Error accessing secret: {e}")
        return None


In [4]:
# ── Project / GCS / BQ configuration ────────────────────────────────────────
PROJECT_ID    = "converged-brandradar-poc"
DATASET_ID    = "brandradar"
REGION        = "europe-west2"
SECRET_ID     = "GEMINI_API_KEY"
BUCKET_NAME   = "converged-brandradar"

# BigQuery table for results (created automatically if it doesn't exist)
BQ_TABLE_ID   = f"{PROJECT_ID}.{DATASET_ID}.CultureConnects"
BQ_WRITE_MODE = "append"   # "append" | "overwrite"

# ── API / model ──────────────────────────────────────────────────────────────
api_key = access_secret_version(PROJECT_ID, SECRET_ID)
if api_key:
    print("✓ API Key retrieved successfully.")
    # With the correct import of google.generativeai, genai.configure is now available.
    genai.configure(api_key=api_key)
else:
    print("❌ Failed to retrieve API key.")

MODELNAME = "models/gemini-3.1-pro-preview"
# Model instantiation should now work correctly after genai.configure.
model     = genai.GenerativeModel(MODELNAME)

# ── GCS / BQ clients ─────────────────────────────────────────────────────────
storage_client = storage.Client()
bq_client      = bigquery.Client(project=PROJECT_ID, location=REGION)

template_path_b = f"gs://{BUCKET_NAME}/templates/CultureScan_template.xlsx"
print(f"✓ Using model: {MODELNAME}")

✓ API Key retrieved successfully.
✓ Using model: models/gemini-3.1-pro-preview


**Load Data**

In [5]:
data = {
    "superspace": [
        "Body & Wellness",
        "Community & Society",
        "Entertainment & Story",
        "Food & Drink",
        "Home & Away",
        "Learning & Creation",
        "Sport & Play",
        "Style & Aesthetics",
    ],
    "activities": [
        "Alternative/Spiritual Wellness,Fitness/Training,Mental Health & Mindfulness,Nutrition/Biohacking,Outdoor Wellness,Recovery/Sleep coaching",
        "Activism,Faith/Spirituality,Local Scenes,Politics,Volunteering",
        "Books & Comics,Film/TV/Streaming,Gaming & Esports,Live Performance (theatre/comedy),Music & Nightlife,Podcasts/Audio",
        "Baking,Coffee/Tea,Dining Out,Home Cooking,Markets & Pop-ups,No/Low Alcohol,Specialty Cuisines",
        "DIY/Maker,Gardening,Household Rituals,Interiors,Pets,Smart Home",
        "Arts & Crafts,Coding/Maker/3D,Languages,Online Learning,Photography/Filmmaking,Writing & Publishing",
        "Dance,Martial Arts,Outdoor/Adventure,Tabletop/Board Games,Team & Individual Sports",
        "Beauty/Grooming,Body Art (tattoos/piercings),Collectibles/Luxury,Design/Architecture,Fashion,Streetwear",
    ],
}
dfsuperspace = pd.DataFrame(data)


In [6]:
REPORTNAME = "CVS Pharmacy"
print("\n📊 Loading base data...")
dfbrands = pd.read_csv(f"gs://{BUCKET_NAME}/lookups/csCVS.csv")

# Optional filters:
# dfbrands = dfbrands.iloc[13:]
# dfbrands = dfbrands.head(1)
# dfbrands = dfbrands[dfbrands["market"].isin(["UK", "USA"])]

print(dfbrands.head(20))
print(f"✓ Loaded {len(dfbrands)} records")
print(f"  Markets:  {dfbrands['market'].unique().tolist()}")
print(f"  Audience: {dfbrands['audience'].unique().tolist()}")



📊 Loading base data...
  market     audience
0    USA  New Parents
✓ Loaded 1 records
  Markets:  ['USA']
  Audience: ['New Parents']


**Core Functions**

In [7]:
def extract_json_robust(response_text):
    """
    Robust JSON extraction with multiple fallback strategies.
    Returns parsed JSON dict or None if all strategies fail.
    """
    if not response_text:
        return None

    # Strategy 1: JSON wrapped in markdown code block
    match = re.search(r'```json\s*(\{.*?\})\s*```', response_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    # Strategy 2: Any JSON object in the text
    match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    # Strategy 3: Entire response is JSON
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass

    return None


In [8]:
def validate_json_fields(json_data, required_fields):
    """Validate that JSON contains all required fields. Returns (is_valid, missing)."""
    if json_data is None:
        return False, required_fields
    missing = [f for f in required_fields if f not in json_data]
    return len(missing) == 0, missing


In [9]:
def safe_get(data, key, default=None):
    """Safely retrieve a key from a dict with a default fallback."""
    return data.get(key, default) if isinstance(data, dict) else default


**Retry Decorator**

In [10]:
def retry_with_backoff(retries=5, backoff_in_seconds=1):
    """Decorator for retrying functions with exponential back-off on quota errors."""
    def rwb(f):
        def wrapper(*args, **kwargs):
            _retries, _backoff = retries, backoff_in_seconds
            while _retries > 1:
                try:
                    return f(*args, **kwargs)
                except exceptions.ResourceExhausted:
                    sleep = _backoff + random.uniform(0, 1)
                    print(f"  ⚠️  API quota exceeded. Retrying in {sleep:.2f}s...")
                    time.sleep(sleep)
                    _backoff *= 2
                    _retries -= 1
                except Exception as e:
                    print(f"  ⚠️  Error: {e}. Retrying in {_backoff}s...")
                    time.sleep(_backoff)
                    _backoff *= 2
                    _retries -= 1
            return f(*args, **kwargs)
        return wrapper
    return rwb


**Setup DataFrames**

In [11]:
def setscandf():
    """Initialise an empty DataFrame for scan results with correct dtypes."""
    df = pd.DataFrame(columns=[
        "Market", "Audience", "SuperSpace",
        "Activity", "InterestLevel", "WeeklyTimeSpenthours", "AnnualSpendDollars",
        "InterestLevelMarket", "WeeklyTimeSpentMarket", "AnnualSpendMarket",
        "Rationale", "Date", "Report",
    ])
    for col in ["InterestLevel", "WeeklyTimeSpenthours", "AnnualSpendDollars",
                "InterestLevelMarket", "WeeklyTimeSpentMarket", "AnnualSpendMarket"]:
        df[col] = df[col].astype("float64")
    # Removed: df["date"] = pd.Series(dtype="datetime64[ns]") to avoid duplicate date columns
    return df

**Prompts**

In [12]:
def set_questionscan(market, audience, activities):
    """Build the Gemini prompt for a given market / audience / activity set."""
    return f"""For {audience} in the {market} market
I need an estimation of the interest level (on a scale 1-10), the average monthly time spend (in hours),
the annual spend (in US Dollars), and a short ~100-word description of your rationale for the scores
for the following activities:
  {activities}

Provide ONLY a JSON object with a key "data" containing an array of objects.
Each object must use these exact keys:
  activity_name          (string)
  interest_level         (number 1-10)
  weekly_time_hours      (float)
  annual_spend_usd       (float)
  interest_level_market  (number 1-10)
  weekly_time_hours_market (float)
  annual_spend_usd_market  (float)
  rationale              (string, ~100 words)
"""


**API Query Function**

In [13]:
@retry_with_backoff(retries=5, backoff_in_seconds=2)
def runQuery(question):
    """
    Query Gemini API.  Quota / transient errors are handled by the decorator.
    Returns response text, or None if the request was blocked.
    """
    response = model.generate_content(question)
    if not response.candidates:
        print(f"  ⚠️  Request blocked. Reason: {response.prompt_feedback}")
        return None
    return response.text


**Data Processing**

In [14]:
def add_to_dataframescan(df, market, audience, superspace, response_text):
    """
    Parse API response and append rows to the results DataFrame.
    Collects all new rows into a list before a single pd.concat call.
    """
    brand_json = extract_json_robust(response_text)
    now        = datetime.now()

    def error_row(msg):
        return {
            "Market": market, "Audience": audience, "SuperSpace": superspace,
            "Activity": "Parse Error",
            "Interest": None, "TimeSpent": None, "MoneySpent": None,
            "InterestMarket": None, "TimeSpentMarket": None, "MoneySpentMarket": None,
            "Description": msg, "date": now, "projectname": REPORTNAME,
        }

    if brand_json is None:
        print(f"  ❌ Failed to extract JSON for '{superspace}' / '{market}'")
        return pd.concat([df, pd.DataFrame([error_row("Failed to extract JSON")])], ignore_index=True)

    data_list = safe_get(brand_json, "data")
    if not isinstance(data_list, list):
        print(f"  ❌ JSON 'data' field is not a list for '{superspace}'")
        return pd.concat([df, pd.DataFrame([error_row("JSON 'data' field is not a list")])], ignore_index=True)

    new_rows = []
    for item in data_list:
        try:
            new_rows.append({
                "Market":          market,
                "Audience":        audience,
                "SuperSpace":      superspace,
                "Activity":        safe_get(item, "activity_name", ""),
                "InterestLevel":        safe_get(item, "interest_level", 0),
                "WeeklyTimeSpenthours":       safe_get(item, "weekly_time_hours", 0),
                "AnnualSpendDollars":      safe_get(item, "annual_spend_usd", 0),
                "InterestLevelMarket":  safe_get(item, "interest_level_market", 0),
                "WeeklyTimeSpentMarket": safe_get(item, "weekly_time_hours_market", 0),
                "AnnualSpendMarket":safe_get(item, "annual_spend_usd_market", 0),
                "Rationale":     safe_get(item, "rationale", ""),
                "Date":            now,
                "Report":     REPORTNAME,
            })
        except Exception as e:
            print(f"  ⚠️  Error processing item: {e}")

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
        print(f"  ✓ Processed {len(new_rows)} activities for '{superspace}'")
    else:
        print("  ⚠️  No valid rows created from response")

    return df



**Batch Processing Functions**

In [15]:
def runscan(dfbrands, df, start_idx=0):
    """Process culture scan for all brand/market records with progress tracking."""
    all_response_data = []
    failed_records    = []
    total             = len(dfbrands)

    print(f"\n{'='*60}")
    print(f"CULTURE SCAN ANALYSIS — Processing {total} records")
    print(f"{'='*60}\n")

    for idx, row in dfbrands.iterrows():
        if idx < start_idx:
            continue

        print(f"[{idx+1}/{total}] {row['market']} — {row['audience']}")

        for _, rowss in dfsuperspace.iterrows():
            superspace = rowss["superspace"]
            try:
                question            = set_questionscan(row["market"], row["audience"], rowss["activities"])
                gemini_text_response = runQuery(question)

                if not gemini_text_response:
                    print(f"  ❌ No response from API for '{superspace}'")
                    gemini_text_response = '{"error": "No response received"}'
                    failed_records.append({
                        "index": idx, "market": row["market"],
                        "audience": row["audience"], "superspace": superspace,
                        "error": "No API response",
                    })

                df = add_to_dataframescan(df, row["market"], row["audience"], superspace, gemini_text_response)

                all_response_data.append({
                    "timestamp":       datetime.now().isoformat(),
                    "market":          row["market"],
                    "audience":        row["audience"],
                    "superspace":      superspace,
                    "input_question":  question,
                    "gemini_response": gemini_text_response,
                    "success":         extract_json_robust(gemini_text_response) is not None,
                })

                time.sleep(1)   # polite rate-limiting between API calls

            except Exception as e:
                print(f"  ❌ CRITICAL ERROR: {e}")
                failed_records.append({
                    "index": idx, "market": row["market"],
                    "audience": row["audience"], "superspace": superspace,
                    "error": str(e),
                })

        # Checkpoint log every 10 outer records
        if (idx + 1) % 10 == 0:
            print(f"  💾 Checkpoint: {idx+1} records processed")

    if failed_records:
        print(f"\n⚠️  {len(failed_records)} calls failed:")
        for rec in failed_records:
            print(f"  - {rec['superspace']} ({rec['market']}, {rec['audience']}): {rec['error']}")

    return all_response_data, df


**Output Functions**

In [16]:
def writejson(all_response_data, repname):
    """Save raw API responses to a JSON file in GCS."""
    destination = f"results/{repname}_{REPORTNAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    try:
        blob = storage_client.bucket(BUCKET_NAME).blob(destination)
        blob.upload_from_string(
            data=json.dumps(all_response_data, indent=4, ensure_ascii=False),
            content_type="application/json",
        )
        print(f"  ✓ JSON saved to gs://{BUCKET_NAME}/{destination}")
    except Exception as e:
        print(f"  ❌ Error saving JSON to GCS: {e}")


In [17]:
def writexls(df, repname, report_name, template_path):
    """Write results DataFrame to an Excel file in GCS using the report template."""
    outname_gcs = f"gs://{BUCKET_NAME}/results/{repname}_{REPORTNAME}_{datetime.now().strftime('%Y%m%d')}.xlsx"
    try:
        # Download template
        tpl_blob_path = "/".join(template_path.split("/")[3:])
        in_mem_tpl = io.BytesIO()
        storage_client.bucket(BUCKET_NAME).blob(tpl_blob_path).download_to_file(in_mem_tpl)
        in_mem_tpl.seek(0)

        book  = load_workbook(in_mem_tpl)
        book["Key"].cell(row=3, column=1, value=report_name)

        sheet = book["Culture Scan"]
        for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=False), 2):
            for c_idx, value in enumerate(row, 1):
                sheet.cell(row=r_idx, column=c_idx, value=value)

        out_mem = io.BytesIO()
        book.save(out_mem)
        out_mem.seek(0)

        out_blob_path = "/".join(outname_gcs.split("/")[3:])
        storage_client.bucket(BUCKET_NAME).blob(out_blob_path).upload_from_file(
            out_mem,
            content_type="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        )
        print(f"  ✓ Excel saved to {outname_gcs}")
    except Exception as e:
        print(f"  ❌ Error creating Excel file: {e}")


In [18]:
def write_to_bigquery(df, table_id=BQ_TABLE_ID, write_mode=BQ_WRITE_MODE):
    """
    Write the results DataFrame to a BigQuery table.

    Args:
        df         : The results DataFrame produced by runscan().
        table_id   : Fully-qualified BQ table  (project.dataset.table).
        write_mode : "append"    — add rows to existing table.
                     "overwrite" — truncate & replace.
    """
    if df.empty:
        print("  ☢️  DataFrame is empty — skipping BigQuery write.")
        return

    disposition_map = {
        "append":    bigquery.WriteDisposition.WRITE_APPEND,
        "overwrite": bigquery.WriteDisposition.WRITE_TRUNCATE,
    }
    write_disp = disposition_map.get(write_mode, bigquery.WriteDisposition.WRITE_APPEND)

    # Column mapping from DataFrame to BigQuery schema
    column_mapping = {
        "Market": "market",
        "Audience": "audience",
        "SuperSpace": "superspace",
        "Activity": "activity",
        "InterestLevel": "InterestLevel",
        "WeeklyTimeSpenthours": "WeeklyTimeSpenthours",
        "AnnualSpendDollars": "AnnualSpendDollars",
        "InterestLevelMarket": "InterestLevelMarket",
        "WeeklyTimeSpentMarket": "WeeklyTimeSpentMarket",
        "AnnualSpendMarket": "AnnualSpendMarket",
    }

    # Explicit schema keeps BQ types clean and avoids OBJECT → STRING surprises
    schema = [
    bigquery.SchemaField("market", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("audience", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("superspace", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("activity", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("InterestLevel", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("WeeklyTimeSpenthours", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("AnnualSpendDollars", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("InterestLevelMarket", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("WeeklyTimeSpentMarket", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("AnnualSpendMarket", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("Rationale", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("Date", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("Report", "STRING", mode="NULLABLE"),
    ]

    # Convert DataFrame to BQ-compatible types and rename columns
    df_bq = df.copy()
    df_bq = df_bq.rename(columns=column_mapping)

    # Ensure correct data types for BigQuery
    for col in [
        "InterestLevel",
        "WeeklyTimeSpenthours",
        "AnnualSpendDollars",
        "InterestLevelMarket",
        "WeeklyTimeSpentMarket",
        "AnnualSpendMarket",
    ]:
        if col in df_bq.columns:
            df_bq[col] = pd.to_numeric(df_bq[col], errors="coerce").astype(float)

    # Ensure timestamp format
    if "Date" in df_bq.columns:
        df_bq["Date"] = pd.to_datetime(df_bq["Date"])

    job_config = bigquery.LoadJobConfig(schema=schema, write_disposition=write_disp)

    try:
        job = bq_client.load_table_from_dataframe(df_bq, table_id, job_config=job_config)
        job.result()  # Wait for the job to complete.
        print(f"  ✓ Loaded {job.output_rows} rows into {table_id}")
    except Exception as e:
        print(f"  ❌ Error writing to BigQuery: {e}")

**Main Execution**

In [19]:
def main():
    """Orchestrate the full Culture Scan pipeline."""
    print(f"\n{'='*60}")
    print("CONFIGURATION SUMMARY")
    print(f"{'='*60}")
    print(f"Report Name      : {REPORTNAME}")
    print(f"Model            : {MODELNAME}")
    print(f"Records to process: {len(dfbrands)}")
    print(f"BQ Table         : {BQ_TABLE_ID}  (mode: {BQ_WRITE_MODE})")
    print(f"{'='*60}\n")

    df_bespoke       = setscandf()
    all_response_data, df_bespoke = runscan(dfbrands, df_bespoke)

    print("\n📝 Saving outputs...")
    writejson(all_response_data, "CultureScan")
    writexls(df_bespoke, "CultureScan", "Culture Scan", template_path_b)
    write_to_bigquery(df_bespoke)

    # Success-rate metric (skip rows flagged as Parse Error)
    valid_rows = (df_bespoke["Activity"] != "Parse Error").sum()
    print(f"\n✅ Culture Scan complete!")
    print(f"   Rows written : {len(df_bespoke)}")
    print(f"   Valid rows   : {valid_rows} / {len(df_bespoke)}")

    end_time = time.time()
    duration = end_time - start_time
    print(f"\n{'='*60}")
    print("PROCESS COMPLETE")
    print(f"{'='*60}")
    print(f"Total runtime  : {duration:.2f}s ({duration/60:.2f} min)")
    print(f"End time       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    if len(dfbrands):
        print(f"Avg per record : {duration/len(dfbrands):.2f}s")
    print(f"{'='*60}\n")
    print("🎉 All operations completed successfully!")


**Helper Functions**

In [20]:
def listmodels():
    """List available Gemini models."""
    try:
        models = genai.list_models()
        print("\nAvailable Gemini Models:")
        print("-" * 60)
        for m in models:
            print(f"  • {m.name}")
    except Exception as e:
        print(f"Error listing models: {e}")


In [21]:
def resume_from_checkpoint(checkpoint_index):
    """
    Resume processing from a specific record index after an interruption.

    Args:
        checkpoint_index: dfbrands row index to start from.
    """
    print(f"\n🔄 Resuming analysis from index {checkpoint_index}")
    df = setscandf()
    all_response_data, df = runscan(dfbrands, df, start_idx=checkpoint_index)
    writejson(all_response_data, "CultureScan_Resumed")
    writexls(df, "CultureScan_Resumed", f"{REPORTNAME}_Resumed", template_path_b)
    write_to_bigquery(df)
    print("\n✅ Resume operation complete!")
    return df


In [22]:
def view_results_summary(df):
    """Display a quick summary of the results DataFrame."""
    print(f"\n{'='*60}")
    print("RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Total rows : {len(df)}")

    errors  = (df["Activity"] == "Parse Error").sum()
    success = len(df) - errors
    print(f"Valid rows : {success}")
    print(f"Errors     : {errors}")
    if len(df):
        print(f"Success %%  : {success/len(df)*100:.1f}%%")

    print("\nMarkets covered:")
    for market, count in df["Market"].value_counts().items():
        print(f"  • {market}: {count}")

    print("\nSuper-spaces covered:")
    for ss, count in df["SuperSpace"].value_counts().items():
        print(f"  • {ss}: {count}")
    print(f"{'='*60}\n")


In [23]:

if __name__ == "__main__":
    main()

    # Uncomment for debugging:
    # listmodels()
    # resume_from_checkpoint(5)



CONFIGURATION SUMMARY
Report Name      : CVS Pharmacy
Model            : models/gemini-3.1-pro-preview
Records to process: 1
BQ Table         : converged-brandradar-poc.brandradar.CultureConnects  (mode: append)


CULTURE SCAN ANALYSIS — Processing 1 records

[1/1] USA — New Parents


/tmp/ipykernel_154/745927291.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)


  ✓ Processed 6 activities for 'Body & Wellness'
  ✓ Processed 5 activities for 'Community & Society'
  ✓ Processed 6 activities for 'Entertainment & Story'
  ✓ Processed 7 activities for 'Food & Drink'
  ✓ Processed 6 activities for 'Home & Away'
  ✓ Processed 6 activities for 'Learning & Creation'
  ✓ Processed 5 activities for 'Sport & Play'
  ✓ Processed 6 activities for 'Style & Aesthetics'

📝 Saving outputs...
  ✓ JSON saved to gs://converged-brandradar/results/CultureScan_CVS Pharmacy_20260413_075221.json
  ✓ Excel saved to gs://converged-brandradar/results/CultureScan_CVS Pharmacy_20260413.xlsx
  ✓ Loaded 47 rows into converged-brandradar-poc.brandradar.CultureConnects

✅ Culture Scan complete!
   Rows written : 47
   Valid rows   : 47 / 47

PROCESS COMPLETE
Total runtime  : 281.34s (4.69 min)
End time       : 2026-04-13 07:52:26
Avg per record : 281.34s

🎉 All operations completed successfully!
